# German Traffic Sign Recognition 

<img src='https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRZT7nuVuwq638pAZe4v9HL2obv8ETmddGNelYS0DVx8A&s=10'>

This project aims to build a deep learning-based Convolutional Neural Network (CNN) to accurately classify and recognize various German traffic signs from images for deployment in web applications.

In [1]:
# Getting datasets
import os
import numpy as np
import pandas as pd
from PIL import Image
train_df=pd.read_csv('Train.csv')

In [2]:
train_df.sample(3)

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
29563,48,45,5,5,42,39,25,Train/25/00025_00036_00013.png
7642,63,63,5,5,58,58,4,Train/4/00004_00039_00022.png
32927,43,42,5,5,38,37,32,Train/32/00032_00002_00017.png


In [3]:
train_df.shape

(39209, 8)

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39209 entries, 0 to 39208
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Width    39209 non-null  int64 
 1   Height   39209 non-null  int64 
 2   Roi.X1   39209 non-null  int64 
 3   Roi.Y1   39209 non-null  int64 
 4   Roi.X2   39209 non-null  int64 
 5   Roi.Y2   39209 non-null  int64 
 6   ClassId  39209 non-null  int64 
 7   Path     39209 non-null  object
dtypes: int64(7), object(1)
memory usage: 2.4+ MB


In [5]:
train_df.isnull().sum()

Width      0
Height     0
Roi.X1     0
Roi.Y1     0
Roi.X2     0
Roi.Y2     0
ClassId    0
Path       0
dtype: int64

In [6]:
train_df['ClassId'].unique() #Has 43 different traffic sign

array([20,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
       16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42])

In [7]:
paths = train_df['Path'].values
labels = train_df['ClassId'].values

In [8]:
image_data = [np.array(Image.open(p).resize((30, 30))) for p in paths]

In [9]:
x = np.array(image_data)
y = np.array(labels)

In [10]:
# Split into Training (80%) and Validation (20%) sets
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
# Normalize pixel values to the range [0, 1]
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0

In [12]:
# Phase 2 - Training the model
import tensorflow as tf
from tensorflow.keras import layers, models

In [13]:
n_classes = 43 #sign classes

In [14]:
# CNN Model
model = models.Sequential([
    # 1. Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(30, 30, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    # 2. Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),
    
    # Fully Connected (Dense) Layers
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(n_classes, activation='softmax') # 
])

C:\Users\asus\anaconda3\envs\dml_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 26, 26, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 11, 11, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 9, 9, 64)            │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 43)                  │          11,051 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 339,019 (1.29 MB)

 Trainable params: 339,019 (1.29 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(x_train, y_train, batch_size=64, epochs=15, validation_data=(x_val, y_val))

Epoch 1/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.5767 - loss: 1.4760 - val_accuracy: 0.9399 - val_loss: 0.2605
Epoch 2/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 26ms/step - accuracy: 0.9111 - loss: 0.2816 - val_accuracy: 0.9833 - val_loss: 0.0723
Epoch 3/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 26ms/step - accuracy: 0.9526 - loss: 0.1541 - val_accuracy: 0.9918 - val_loss: 0.0357
Epoch 4/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 12s 25ms/step - accuracy: 0.9658 - loss: 0.1122 - val_accuracy: 0.9901 - val_loss: 0.0337
Epoch 5/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 26ms/step - accuracy: 0.9742 - loss: 0.0844 - val_accuracy: 0.9932 - val_loss: 0.0242
Epoch 6/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.9773 - loss: 0.0726 - val_accuracy: 0.9955 - val_loss: 0.0195
Epoch 7/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.9830 - loss: 0.0587 - val_accuracy: 0.9941 - val_loss: 0.0207
Epoch 8/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - accuracy: 0.9822 - loss: 0.0559 - 

In [17]:
model.save("traffic_sign_model.h5")